[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Análisis exploratorio de datos completo con `mpg`.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo IV: *Proyecto integrador*

En este *notebook* se recorrerá de principio a fin un flujo de EDA reproducible sobre `mpg`: pregunta, auditoría de calidad, limpieza, distribuciones, relaciones, una prueba con tamaño del efecto y conclusiones.

## 📚 Referencias.

Este *notebook* integra lo visto en las sesiones 1 a 6. Para la API: [py311 · 20 Limpieza y datos faltantes](https://github.com/PythonistaMX/py311/blob/main/20_limpieza_y_datos_faltantes.ipynb), [py311 · 24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb), [py311 · 31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb) y la documentación de [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway

---
## 1. Estructura de un EDA (10 min).

| Paso | Pregunta guía | Producto |
|---|---|---|
| 1. Pregunta | ¿Qué se quiere saber y para qué decisión? | 2 o 3 preguntas respondibles con los datos |
| 2. Datos | ¿De dónde vienen, quién los recolectó, qué población representan? | Diccionario de datos y sesgos posibles |
| 3. Calidad | ¿Duplicados, faltantes, tipos, rangos, unidades? | Auditoría y decisiones documentadas |
| 4. Descripción | ¿Cómo se distribuye cada variable clave? | Medidas adecuadas a la forma; atípicos discutidos |
| 5. Relaciones | ¿Qué se asocia con qué? ¿Hay confusores? | Tablas y gráficas por grupo |
| 6. Inferencia | ¿Las diferencias superan la variación muestral? ¿Qué tan grandes son? | Pruebas con supuestos, IC y tamaño del efecto |
| 7. Conclusiones | ¿Qué responde a cada pregunta y con qué límites? | Hallazgos en lenguaje llano, sin causalidad injustificada |

El orden no es estrictamente lineal: lo que aparece en el paso 5 suele obligar a volver al 3 o a reformular el 1.

**¿Por qué una estructura?** Un EDA sin estructura se vuelve una colección de gráficas sin conclusión. Cada paso responde a una pregunta y produce algo concreto que se usa en el siguiente: las preguntas deciden qué variables auditar; la auditoría decide qué datos se usan; la descripción decide qué medidas y pruebas son adecuadas; las relaciones revelan confusores que cambian la interpretación de las pruebas. Las sesiones 1 a 6 cubrieron cada paso por separado; hoy se encadenan.

### Funciones reutilizables.

En lugar de copiar y pegar celdas, conviene definir funciones pequeñas una vez y aplicarlas a cualquier *dataset*. Estas dos sirven para el proyecto.

* La siguiente celda definirá `perfil_rapido` (una fila por columna con tipo, faltantes, únicos y ejemplo) y `resumen_por_grupo` (n, media, mediana, desviación estándar e IQR por grupo).

In [ ]:
def perfil_rapido(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_faltantes": df.isna().sum(),
        "pct_faltantes": (df.isna().mean() * 100).round(1),
        "n_unicos": df.nunique(dropna=True),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


def resumen_por_grupo(df, columna, grupo):
    return df.groupby(grupo, observed=True)[columna].agg(
        n="count", media="mean", mediana="median", desv_est="std",
        iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
    ).round(2)

**¿Qué hace cada función?** `perfil_rapido` reúne en una tabla lo que en la sesión 2 se revisaba con `info()`, `isna()` y `nunique()`. `resumen_por_grupo` da, por grupo, el *n* (para no comparar grupos diminutos), un centro robusto y otro sensible (mediana y media) y dos dispersiones (desviación estándar e IQR). Ambas se aplicarán sin cambios al *dataset* del proyecto.

---
## 2. Demostración en vivo con `mpg` (60 min).

### 2.1 Pregunta y datos.

`mpg` (*Auto MPG*, repositorio UCI, incluido en *Seaborn*) describe 398 modelos de automóviles vendidos en EE. UU. entre 1970 y 1982: rendimiento (`mpg`, millas por galón estadounidense), cilindros, desplazamiento (pulgadas cúbicas), potencia (`horsepower`), peso (libras), aceleración (segundos de 0 a 60 mph), año del modelo, origen y nombre.

**Preguntas de análisis:**
1. ¿Qué características del auto se asocian más con el rendimiento?
2. ¿Difiere el rendimiento según el origen (EE. UU., Europa, Japón)? ¿Esa diferencia se explica por otras características?

**Población y sesgos posibles:** cada fila es un **modelo**, no un auto vendido: un modelo que vendió un millón de unidades pesa lo mismo que uno raro. Las conclusiones se refieren a "los modelos ofrecidos", no a "los autos en circulación". Además, solo incluye el mercado de EE. UU. de esa época.

* La siguiente celda leerá `mpg.csv` y mostrará su perfil rápido.

In [ ]:
autos = pd.read_csv(os.path.join(DATA, "mpg.csv"))
autos.shape, perfil_rapido(autos)

**Interpretación:** 398 filas y 9 columnas. Solo `horsepower` tiene faltantes (6, 1.5%). `origin` y `name` son texto (`str`); `cylinders` y `model_year` son enteros con pocos valores distintos (5 y 13): son **discretas**, y `cylinders` podría tratarse como ordinal. `name` tiene 305 valores únicos para 398 filas, así que **no** es un identificador: hay que averiguar por qué se repite.

### 2.2 Auditoría de calidad.

* La siguiente celda contará duplicados exactos, nombres repetidos y repeticiones de la combinación nombre y año.

In [ ]:
{"duplicados exactos": int(autos.duplicated().sum()),
 "nombres repetidos": int(autos["name"].duplicated().sum()),
 "nombre + año repetidos": int(autos.duplicated(subset=["name", "model_year"]).sum())}

**Interpretación:** no hay filas idénticas, pero 93 nombres se repiten y 2 combinaciones de nombre y año también. Antes de eliminar nada se **miran** los casos: un duplicado parcial puede ser un error de captura o dos observaciones legítimas.

* La siguiente celda mostrará los dos casos con el mismo nombre y año.

In [ ]:
autos[autos.duplicated(subset=["name", "model_year"], keep=False)].sort_values("name")

**Decisión:** un mismo nombre en años distintos es el mismo modelo en otra versión (no es un duplicado). Los dos pares con mismo nombre y año tienen pesos y rendimientos distintos: son versiones diferentes (motor o equipamiento). Se **conservan**; la combinación nombre-año no es una llave confiable en este *dataset*.

* La siguiente celda mostrará los autos sin `horsepower` y la frecuencia de cada número de cilindros.

In [ ]:
autos.loc[autos["horsepower"].isna(), ["name", "model_year", "cylinders", "displacement", "weight"]], autos["cylinders"].value_counts().sort_index()

**Interpretación:** los 6 autos sin potencia son de años variados (1971 a 1982), casi todos de 4 cilindros y de peso ligero a medio (Ford, Renault y AMC); no se ve un patrón que haga pensar que faltan por una razón relacionada con el consumo (MCAR plausible, sesión 2). En los cilindros, 3 y 5 tienen 4 y 3 modelos: grupos tan pequeños dan medias inestables.

* La siguiente celda extraerá la marca (primera palabra de `name`) y mostrará sus frecuencias para buscar variantes mal escritas.

In [ ]:
marca = autos["name"].str.split().str[0]
marca.value_counts().sort_index().to_dict()

**Interpretación:** la marca, una variable derivada del texto libre `name`, revela erratas (`chevroelt`, `toyouta`, `vokswagen`) y equivalencias (`chevy` y `chevrolet`, `vw` y `volkswagen`). Es un ejemplo típico de por qué el texto libre se audita antes de usarlo como categoría.

**Decisiones de calidad (documentadas):**
* `horsepower`: faltan 6 de 398 valores (1.5%), de modelos variados. Con tan pocos faltantes, eliminar o imputar casi no cambia los resultados; se usará `dropna(subset=["horsepower"])` **solo** en los análisis que usan esa variable, para no perder filas en los demás.
* `cylinders`: 3 y 5 cilindros tienen 4 y 3 modelos. No son errores (los motores rotativos de Mazda se codificaron como 3 cilindros; los de 5 son Audi y Mercedes-Benz), pero son grupos demasiado pequeños para compararlos; en comparaciones por cilindros se usarán solo 4, 6 y 8.
* Marcas: hay variantes mal escritas o equivalentes (`chevroelt`, `chevy` y `chevrolet`; `maxda` y `mazda`; `toyouta`; `vokswagen`, `vw` y `volkswagen`; `mercedes` y `mercedes-benz`). No se usan en las preguntas de hoy, así que no se corrigen; si una pregunta usara la marca, habría que unificarlas con un mapa explícito (sesión 2).
* `origin` se convierte a `category`. Los rangos de todas las variables son plausibles (`describe` no muestra imposibles).

* La siguiente celda aplicará las decisiones de tipo y creará la tabla `autos_limpio`, sin modificar los datos originales.

In [ ]:
autos_limpio = autos.astype({"origin": pd.CategoricalDtype(["usa", "europe", "japan"])})
autos_limpio.describe().T.round(1)

**Interpretación:** `describe` no muestra valores imposibles: rendimientos de 9 a 46.6 mpg, pesos de 1,613 a 5,140 lb y años de 70 a 82. Guardar el resultado en `autos_limpio` mantiene intacto `autos`, así cualquier decisión se puede revisar.

### 2.3 Distribuciones y una decisión de escala.

`mpg` es una razón "distancia por combustible". Para comparar consumo suele ser más natural la escala inversa, **litros por cada 100 km** (`235.215 / mpg`): ahorrar 10 mpg significa mucho más combustible al pasar de 10 a 20 mpg que de 30 a 40. Elegir la escala cambia la forma de las relaciones.

**Concepto: transformar la escala.** Pasar de mpg a litros por cada 100 km no es solo un cambio de unidades: es una transformación **no lineal**, que separa los valores bajos de mpg y junta los altos (pasar de 10 a 20 mpg ahorra 11.8 L por cada 100 km; pasar de 30 a 40, solo 2). Una relación que es una hipérbola en una escala puede ser una recta en la otra. En el EDA, elegir la escala en la que las relaciones se ven lineales simplifica la descripción (una correlación de Pearson basta) y hace válidos los métodos lineales. Es la misma idea que usar logaritmos con variables asimétricas (sesión 3).

* La siguiente celda creará `litros_100km` y comparará la distribución de ambas escalas.

In [ ]:
autos_limpio = autos_limpio.assign(litros_100km=lambda d: 235.215 / d["mpg"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
sns.histplot(autos_limpio, x="mpg", bins=30, ax=axes[0]).set(xlabel="rendimiento (millas por galón)", title="mpg")
sns.histplot(autos_limpio, x="litros_100km", bins=30, ax=axes[1]).set(xlabel="consumo (L/100 km)", title="L/100 km")
plt.tight_layout()
plt.show()

**Interpretación:** las dos distribuciones son asimétricas a la derecha, pero en distinto grado: la mitad central de mpg va de 17.5 a 29 (asimetría ≈ 0.46) y la de L/100 km de 8.1 a 13.4 (asimetría ≈ 0.77), con una cola hacia los autos que consumen mucho. La forma depende de la escala elegida, no solo de los datos.

**Concepto: medir la curvatura.** Si la relación es lineal, los residuos de la recta no tienen patrón. Si es curva, los residuos son positivos en los extremos y negativos en el centro (o al revés), así que se relacionan con qué tan lejos está cada auto del peso promedio (esa distancia elevada al cuadrado es grande en los dos extremos y chica en el centro). Esa correlación sirve como indicador sencillo de curvatura: cerca de 0 significa "casi lineal".

* La siguiente celda comparará la relación con el peso en ambas escalas: correlación de Pearson y curvatura (correlación de los residuos de una recta con el peso al cuadrado).

In [ ]:
def curvatura(x, y):
    pendiente, ordenada = np.polyfit(x, y, deg=1)
    residuos = y - (ordenada + pendiente * x)
    return np.corrcoef(residuos, (x - x.mean()) ** 2)[0, 1]

pd.DataFrame({
    escala: {"pearson con el peso": autos_limpio["weight"].corr(autos_limpio[escala]),
             "curvatura": curvatura(autos_limpio["weight"], autos_limpio[escala])}
    for escala in ["mpg", "litros_100km"]
}).round(3)

**Interpretación:** en mpg, la correlación con el peso es −0.832 y la curvatura 0.239; en L/100 km, la correlación es 0.885 y la curvatura baja a 0.108. La misma información se describe mejor con una recta en la escala de consumo.

* La siguiente celda dibujará ambas relaciones con una recta ajustada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, escala, etiqueta in zip(axes, ["mpg", "litros_100km"], ["rendimiento (mpg)", "consumo (L/100 km)"]):
    sns.regplot(autos_limpio, x="weight", y=escala, ax=ax, scatter_kws={"alpha": 0.4}, line_kws={"color": "black"})
    ax.set(xlabel="peso (lb)", ylabel=etiqueta)
axes[0].set_title("En mpg la relación es curva")
axes[1].set_title("En L/100 km es casi lineal")
plt.tight_layout()
plt.show()

**Consecuencia:** con mpg, una recta subestima el rendimiento de los autos muy ligeros y muy pesados (los residuos tienen forma de U), y Pearson (−0.83) subestima la fuerza de la relación (Spearman, que solo pide monotonía, da −0.87). En L/100 km la relación es casi lineal (*r* = 0.89). Se reportará el consumo en L/100 km para las relaciones y mpg cuando se necesite comparar con fuentes de EE. UU.

### 2.4 Relaciones (pregunta 1).

* La siguiente celda calculará la correlación de cada variable numérica con el consumo, ordenada.

In [ ]:
numericas = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
autos_limpio[numericas].corrwith(autos_limpio["litros_100km"]).sort_values(ascending=False).round(3)

**Interpretación:** peso (0.885), desplazamiento (0.867), potencia (0.855) y cilindros (0.839) se asocian fuertemente con más consumo; el año del modelo (−0.557) y la aceleración (−0.456), con menos. Un tiempo de aceleración **mayor** indica un auto más lento, y por eso menos potente. Pero esta tabla no dice si son cuatro hallazgos o uno solo: hay que mirar cómo se correlacionan las características **entre sí**.

* La siguiente celda mostrará la matriz de correlaciones entre las características.

In [ ]:
sns.heatmap(autos_limpio[numericas + ["litros_100km"]].corr(), annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="vlag")
plt.title("Peso, desplazamiento, potencia y cilindros miden casi lo mismo: el tamaño del motor y del auto")
plt.show()

Peso, desplazamiento, potencia y cilindros tienen correlaciones de 0.84 a 0.95 entre sí: son casi **la misma información** (el tamaño del auto). Decir "la potencia aumenta el consumo" y "el peso aumenta el consumo" como si fueran hallazgos independientes sería un error.

### 2.5 Comparación por origen (pregunta 2).

**¿Por qué este paso?** Antes de cualquier prueba se describe cada grupo: tamaño, centro y dispersión. Resumir también el **peso** por origen es deliberado: si una variable fuertemente asociada con el consumo difiere entre orígenes, puede ser un **confusor**, una variable relacionada con el grupo y con el resultado que explica parte de la diferencia observada.

* La siguiente celda resumirá el consumo y el peso por origen.

In [ ]:
pd.concat({"consumo (L/100 km)": resumen_por_grupo(autos_limpio, "litros_100km", "origin"),
           "peso (lb)": resumen_por_grupo(autos_limpio, "weight", "origin")}, axis=1)

**Interpretación:** los 249 modelos de EE. UU. consumen en mediana 12.7 L/100 km, frente a 8.9 de los europeos (70) y 7.4 de los japoneses (79). Pero también pesan en promedio 3,362 lb, frente a 2,423 y 2,221. La desviación estándar del consumo de EE. UU. es mucho mayor: ofrecían desde compactos hasta autos muy grandes.

* La siguiente celda aplicará Levene, el ANOVA de Welch y Tukey al consumo por origen, y calculará η².

In [ ]:
grupos_origen = [g["litros_100km"].to_numpy() for _, g in autos_limpio.groupby("origin", observed=True)]
media_global = autos_limpio["litros_100km"].mean()
por_origen = autos_limpio.groupby("origin", observed=True)["litros_100km"].agg(["count", "mean"])
eta2_origen = (por_origen["count"] * (por_origen["mean"] - media_global) ** 2).sum() / ((autos_limpio["litros_100km"] - media_global) ** 2).sum()
tukey_origen = pairwise_tukeyhsd(autos_limpio["litros_100km"], autos_limpio["origin"].astype(str))
({"Levene p": round(float(stats.levene(*grupos_origen).pvalue), 3),
  "Welch p": float(anova_oneway(autos_limpio["litros_100km"], autos_limpio["origin"], use_var="unequal").pvalue),
  "eta²": round(float(eta2_origen), 3)},
 pd.DataFrame(tukey_origen.summary().data[1:], columns=tukey_origen.summary().data[0]))

Levene rechaza la igualdad de varianzas (los autos de EE. UU. son mucho más variados), por eso se usa el ANOVA de Welch; Tukey supone varianzas iguales, así que sus intervalos se toman como aproximados. La diferencia es significativa y grande: el origen "explica" 30% de la variabilidad del consumo y los autos de EE. UU. consumen unos 4 L/100 km más que los europeos y japoneses. Pero los autos de EE. UU. también pesan unos 1,100 lb más. ¿Es el origen o es el peso?

**Decisión: comparar dentro de niveles del confusor.** Se dividirá el peso en cuartiles y se comparará el consumo por origen **dentro** de cada cuartil.

**Concepto: estratificar por el confusor.** Si el peso explica parte de la diferencia, al comparar autos de **peso parecido** esa parte desaparece. Es la misma idea del muestreo estratificado (sesiones 1 y 5) aplicada al análisis: dentro de cada estrato el confusor casi no varía. `pd.qcut` crea cuatro grupos con aproximadamente el mismo número de modelos. Hay que revisar el *n* de cada celda, porque un estrato sin representantes de un grupo no permite comparar.

* La siguiente celda creará cuartiles de peso y mostrará la mediana del consumo y el número de modelos por origen en cada cuartil.

In [ ]:
autos_limpio = autos_limpio.assign(cuartil_peso=pd.qcut(autos_limpio["weight"], 4, labels=["Q1 ligero", "Q2", "Q3", "Q4 pesado"]))
autos_limpio.pivot_table(index="cuartil_peso", columns="origin", values="litros_100km",
                         aggfunc=["median", "count"], observed=True).round(1)

* La siguiente celda dibujará el consumo contra el peso, coloreado por origen, con una recta por origen.

In [ ]:
grafica = sns.lmplot(autos_limpio, x="weight", y="litros_100km", hue="origin", height=4.5, aspect=1.5,
                     scatter_kws={"alpha": 0.5}, ci=None)
grafica.set_axis_labels("peso (lb)", "consumo (L/100 km)")
grafica.ax.set_title("A igual peso, las diferencias por origen se reducen mucho")
plt.show()

**Interpretación:** en el rango de pesos que comparten los tres orígenes (unas 2,000 a 3,000 lb), las rectas quedan a menos de 1 L/100 km entre sí: a 2,500 lb predicen 9.1 (Europa), 8.9 (Japón) y 9.2 L/100 km (EE. UU.). La recta de EE. UU. es más empinada, así que la brecha crece hacia los autos pesados. Los puntos de EE. UU. se extienden hacia la derecha, donde no hay autos europeos ni japoneses. Una gráfica así muestra a la vez el confusor y la **falta de traslape**.

**Lectura:** dentro de los dos cuartiles más ligeros, las medianas de consumo por origen difieren menos de 1 L/100 km (en el más ligero, los de EE. UU. consumen menos que los europeos), frente a los 4 a 5 L/100 km de la comparación sin controlar el peso. En el tercer cuartil los de EE. UU. sí consumen más (12.3 contra 10.2 a 10.9 L/100 km), pero solo hay 6 japoneses y 16 europeos. Además, el cuartil más pesado tiene 99 modelos de EE. UU., 1 europeo y ningún japonés: para autos pesados **no hay con qué comparar**. La gran diferencia por origen refleja, sobre todo, que EE. UU. ofrecía autos más grandes.

### 2.6 Conclusiones de la demostración.

1. El consumo se asocia fuertemente con el tamaño del auto (peso, motor, potencia y cilindros, que miden casi lo mismo); en L/100 km la relación con el peso es casi lineal (*r* = 0.89).
2. Los modelos de EE. UU. consumen bastante más que los europeos y japoneses (unos 4 a 5 L/100 km más; ANOVA de Welch *p* < 0.001, η² = 0.30), pero a igual peso la diferencia se reduce a menos de 1 L/100 km en los autos ligeros: buena parte se explica por el peso, que actúa como confusor. No hay datos para comparar autos pesados de distinto origen.
3. **Límites:** cada fila es un modelo (no ponderado por ventas), solo mercado de EE. UU. de 1970 a 1982; los resultados son asociaciones observacionales.

**💡 Ideas clave de la sección 2:**
* Auditar antes de analizar: duplicados (exactos y parciales), faltantes, grupos pequeños, texto libre.
* La escala importa: en L/100 km la relación con el peso es casi lineal.
* Varias características correlacionadas pueden ser **un solo** hallazgo.
* Una diferencia grande entre grupos puede deberse a un confusor; se compara dentro de estratos y se revisa el traslape.
* Toda decisión se escribe: qué, por qué y cuántas filas afecta.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `24_reproducibilidad_y_arranque_del_proyecto.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>